# 🔵 Decision Tree Classification
---

## 📑 Table of Contents
1. [Introduction & Definition](#1.-Introduction-&-Definition)
2. [How It Works — Math & Intuition](#2.-How-It-Works-—-Math-&-Intuition)
3. [When To Use](#3.-When-To-Use)
4. [Pros & Cons](#4.-Pros-&-Cons)
5. [Dataset Upload & Exploration](#5.-Dataset-Upload-&-Exploration)
   - 5.1 Import Libraries
   - 5.2 Load Dataset
   - 5.3 Exploratory Data Analysis (EDA)
   - 5.4 Preprocessing & Feature Engineering
6. [Training the Model](#6.-Training-the-Model)
   - 6.1 Train/Validation/Test Split
   - 6.2 Fit the Model
7. [Validation](#7.-Validation)
   - 7.1 Cross-Validation
   - 7.2 Validation Metrics
8. [Testing](#8.-Testing)
   - 8.1 Final Test Set Evaluation
   - 8.2 Confusion Matrix & Classification Report
9. [Optimization / Hyperparameter Tuning](#9.-Optimization-/-Hyperparameter-Tuning)
   - 9.1 Cost Complexity Pruning (Alpha Tuning)
   - 9.2 GridSearchCV for Max Depth & Min Samples
   - 9.3 Best Parameters & Retrain
10. [Visualization](#10.-Visualization)
11. [Tips & Tricks Summary](#11.-Tips-&-Tricks-Summary)
12. [Conclusion](#12.-Conclusion)

## 1. Introduction & Definition

The **Decision Tree** is one of the most intuitive, powerful, and widely used algorithms in Machine Learning. Unlike mathematically rigid algorithms like Logistic Regression that draw strict lines (hyperplanes) through data, a Decision Tree mimics human decision-making by breaking a complex problem down into a series of simple, highly interpretable "Yes/No" questions.

### 🧠 The Core Concept
A Decision Tree builds a flowchart-like structure where:
- **Root Node:** Represents the entire dataset, which is split into two or more homogeneous sets based on the most important feature.
- **Internal Nodes (Splits):** Represent tests on specific features (e.g., "Is age < 30?").
- **Branches:** Represent the outcome of the test.
- **Leaf Nodes:** Represent the final class label or prediction.

Decision Trees are **non-parametric** algorithms, meaning they do not make any underlying assumptions about the distribution of the data (like assuming a linear relationship). They learn highly non-linear decision boundaries by dividing the feature space into distinct rectangular regions.

## 2. How It Works — Math & Intuition

How does the tree know *which* feature to split on first, and exactly *what value* to split at? It relies on mathematical metrics of **Information Theory** to evaluate the "purity" of the resulting split. 

The goal of the algorithm is to take a messy, mixed dataset and split it so that the resulting child groups are as "pure" (homogeneous) as possible. It evaluates this using two primary metrics: **Gini Impurity** and **Entropy (Information Gain)**.

### 2.1 Gini Impurity (Default in Scikit-Learn)
Gini Impurity measures the probability that a randomly chosen element from the set would be incorrectly labeled if it was randomly labeled according to the distribution of labels in the subset.
$$ Gini = 1 - \sum_{i=1}^{c} (p_i)^2 $$
- $p_i$ is the probability of an item belonging to class $i$.
- A Gini score of `0.0` represents perfect purity (all elements belong to exactly one class).
- A Gini score of `0.5` represents maximum impurity (a perfectly random 50/50 split).

### 2.2 Entropy & Information Gain
Entropy is a measure of chaos or randomness in the system, derived from physics and information theory.
$$ Entropy = - \sum_{i=1}^{c} p_i \log_2(p_i) $$
**Information Gain** is the reduction in Entropy after a dataset is split on an attribute. The tree computes the Information Gain for every possible split across every feature and chooses the split that maximizes the Information Gain (i.e., the split that reduces the chaos the most).

### 🚨 The Curse of Overfitting
If left completely unconstrained, a Decision Tree will continue splitting until every single leaf node contains exactly 1 data point (perfect purity). This results in a tree that perfectly memorizes the training data but fails terribly on unseen test data. To prevent this, we use **Pruning** and hyperparameter constraints (like limiting the `max_depth` or increasing `min_samples_leaf`).

## 3. When To Use

**Use Decision Trees when:**
- **Interpretability is critical:** You need to explain the model's exact logic to business stakeholders or regulators (e.g., medical diagnoses, loan approvals). You can print the tree and follow the exact logic path.
- **Data requires minimal prep:** Decision Trees do NOT require feature scaling (standardization/normalization), and they handle categorical features and outliers reasonably well.
- **Non-Linear Relationships:** The data has highly complex, non-linear patterns that standard linear models (like Logistic Regression) cannot capture.
- **Baseline Modeling:** A single tree is an excellent baseline before moving to powerful ensemble methods like Random Forests or Gradient Boosting.

## 4. Pros & Cons

| ✅ Pros | ❌ Cons |
| :--- | :--- |
| **Highly Interpretable:** Easy to visualize and explain to non-technical audiences. | **Prone to Overfitting:** Easily memorizes noise in the training data if not constrained. |
| **No Scaling Required:** Impervious to the scale of features; does not require Standardization. | **High Variance:** Very sensitive to small changes in the training data (a tiny change can result in a completely different tree structure). |
| **Handles Non-Linearity:** Naturally models non-linear relationships without manual feature engineering. | **Greedy Algorithm:** It makes locally optimal splits at each node, which may not lead to the globally optimal tree. |

In [ ]:
# ============================================================
# SECTION 5.1 — IMPORT LIBRARIES
# ============================================================

# Core data manipulation and math
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Scikit-Learn tools
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# SECTION 5.2 — LOAD DATASET (Breast Cancer Wisconsin)
# ============================================================
# We are using the Breast Cancer dataset for classification (predicting Malignant vs Benign tumors based on cellular features).

data = load_breast_cancer()
X_full = pd.DataFrame(data.data, columns=data.feature_names)
y_full = pd.Series(data.target, name='Target')

# 0 = Malignant, 1 = Benign
print(f"Dataset Shape: {X_full.shape}")
display(X_full.head())

In [ ]:
# ============================================================
# SECTION 5.3 & 5.4 — EDA & PREPROCESSING
# ============================================================
# 🚨 CRITICAL TIP: Decision Trees DO NOT require feature scaling! 
# Because the tree splits based on threshold values (e.g., radius > 15), whether the radius is scaled to a z-score or left in raw units, 
# the mathematical split point and purity gain remain absolutely identical.

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x=y_full, ax=ax[0], palette='Set2')
ax[0].set_title('Target Class Distribution')
ax[0].set_xticks([0, 1])
ax[0].set_xticklabels(['Malignant (0)', 'Benign (1)'])

sns.boxplot(x=y_full, y=X_full['mean radius'], ax=ax[1], palette='Set2')
ax[1].set_title('Mean Radius vs Target')
ax[1].set_xticks([0, 1])
ax[1].set_xticklabels(['Malignant (0)', 'Benign (1)'])
plt.show()

In [ ]:
# ============================================================
# SECTION 6.1 — TRAIN/VALIDATION/TEST SPLIT
# ============================================================

# We split the data into 80% training and 20% testing sets.
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# ============================================================
# SECTION 6.2 — FIT THE MODEL (Unconstrained Baseline)
# ============================================================

# We instantiate an entirely unconstrained tree. 
# This tree will grow until every leaf is perfectly pure. This is guaranteed to overfit.
tree_baseline = DecisionTreeClassifier(random_state=42)

# Fit to training data
tree_baseline.fit(X_train, y_train)
print(f"Baseline Tree Depth: {tree_baseline.get_depth()}")
print(f"Baseline Number of Leaves: {tree_baseline.get_n_leaves()}")

In [ ]:
# ============================================================
# SECTION 7.1 & 7.2 — CROSS-VALIDATION
# ============================================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(tree_baseline, X_train, y_train, cv=kf, scoring='accuracy')

print(f"Baseline 5-Fold CV Accuracy: {cv_scores.mean():.4f} (Std: +/- {cv_scores.std():.4f})")

In [ ]:
# ============================================================
# SECTION 8 — TESTING & EVALUATION
# ============================================================

def evaluate_classification(model, name, X, y_true):
    # Make predictions
    y_pred = model.predict(X)
    
    # Calculate core metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"--- {name} Test Performance ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}\n")
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# Test the unconstrained baseline tree
evaluate_classification(tree_baseline, "Unconstrained Tree", X_test, y_test)

In [ ]:
# ============================================================
# SECTION 9 — OPTIMIZATION (Hyperparameter Tuning)
# ============================================================
# We use GridSearchCV to systematically test structural constraints on the tree to prevent overfitting.
# Key parameters:
# max_depth: Prevents the tree from growing too deep.
# min_samples_split: The minimum number of samples required to split an internal node.
# min_samples_leaf: The minimum number of samples required to be at a leaf node (prevents leaves with just 1 sample).

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10]
}

print("Running GridSearchCV...")
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"\n🏆 Best Parameters Found: {grid_search.best_params_}")

# Extract the best model
best_tree = grid_search.best_estimator_
print(f"Optimized Tree Depth: {best_tree.get_depth()}")

# Evaluate the Optimized Tree
evaluate_classification(best_tree, "Optimized Pruned Tree", X_test, y_test)

In [ ]:
# ============================================================
# SECTION 10 — VISUALIZATION (Plotting the Tree)
# ============================================================
# This is the defining feature of a Decision Tree: Total transparency. 
# We can visualize the exact logic path the model uses to make a decision.

plt.figure(figsize=(20, 10))
plot_tree(best_tree, 
          feature_names=data.feature_names,  
          class_names=['Malignant', 'Benign'],
          filled=True, 
          rounded=True, 
          fontsize=10)
plt.title("Visualizing the Optimized Decision Tree", fontsize=16)
plt.show()

print("Look at the Root Node (the top block). The model determined that 'mean concave points' is the single most important feature to split on first.")

## 11. Tips & Tricks Summary

1. **Beware of Overfitting:** An unconstrained Decision Tree is guaranteed to overfit. You must utilize hyperparameters like `max_depth` or `min_samples_leaf` to prune the tree and force it to generalize.
2. **Cost Complexity Pruning:** Scikit-Learn offers an advanced pruning technique via the `ccp_alpha` parameter, which mathematically prunes the weakest links in the tree post-creation.
3. **Feature Importance:** Decision Trees provide an excellent `feature_importances_` attribute, allowing you to instantly see which features drive the model's logic.
4. **Instability:** Decision trees suffer from high variance. Changing a few rows in the training data can result in an entirely different tree structure. This is why we usually graduate from a single Decision Tree to a Random Forest (an ensemble of trees).

## 12. Conclusion

The Decision Tree is a beautiful algorithm that bridges the gap between machine learning math and human-interpretable logic. By mathematically measuring and reducing entropy, it constructs a highly efficient flowchart capable of modeling complex, non-linear patterns. 

While its tendency to overfit makes a solitary Decision Tree somewhat fragile for complex production systems, understanding how to construct, tune, and prune a tree is an absolute prerequisite for mastering modern ensemble techniques like Random Forests and XGBoost.